In [39]:
##### Assembles a shapefile with all sub-national geographies for both capital and labor

import os
import pandas as pd
import geopandas as gpd
import numpy as np

In [40]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())

# Import data
labor = gpd.read_file(f"{cd}/Data/Clean/Geographies/subnational_labor.shp")
capital = gpd.read_file(f"{cd}/Data/Clean/Geographies/subnational_capital.shp")

# Set save path
save_path = f"{cd}/Data/Clean/Geographies/subnational_total.shp"

In [41]:
##### Clean and merge

# Re-name columns 
capital = capital.rename(columns={'value': 'stock_mUSD'})
labor = labor.rename(columns={'value': 'ag_jobs'})

# keep only needed columns 
capital = capital[['ISO3', 'GEO_ID', 'GEO_ID_NM', 'stock_mUSD', 'PROJ_ID', 'geometry']]
labor = labor[['ISO3', 'GEO_ID', 'GEO_ID_NM', 'ag_jobs', 'PROJ_ID', 'geometry']]

# merge 
total = capital.merge(labor, on=['ISO3', 'GEO_ID', 'PROJ_ID', 'GEO_ID_NM'], how='outer')

# fix geometries 
total['geometry'] = total['geometry_x'].fillna(total['geometry_y'])
total = total[['ISO3', 'GEO_ID', 'GEO_ID_NM', 'PROJ_ID', 'stock_mUSD', 'ag_jobs', 'geometry']]

# add column to ID if its for capital/labor/or both
conditions = [
    total['stock_mUSD'].notna() & total['ag_jobs'].isna(),
    total['ag_jobs'].notna() & total['stock_mUSD'].isna(),
    total['ag_jobs'].notna() & total['stock_mUSD'].notna()
]

choices = ['capital', 'labor', 'both']

total['subset'] = np.select(conditions, choices, default=None)

In [42]:
##### Save
total.to_file(save_path)